# Future-Guided Learning — CHB-MIT (Colab)

Tái hiện track EEG của bài báo: Gunasekaran et al., *A predictive approach to enhance
time-series forecasting*, Nature Communications 2025. Repo tác giả:
https://github.com/SkyeGunasekaran/FutureGuidedLearning

**Chạy trên Colab runtime có GPU** (Runtime > Change runtime type > T4 GPU).

Pipeline 3 giai đoạn:
1. Train **teacher** (phát hiện động kinh — `seizure_detection.py`)
2. Train **baseline** (dự báo động kinh, không FGL — `seizure_prediction.py`)
3. Train **student + FGL** (dự báo động kinh, có teacher chỉ dẫn — `FGL_CHBMIT.py`)

Mọi thứ (code, dữ liệu EEG thô, cache đã xử lý, checkpoint) đều lưu trong **Google
Drive** để bền qua các phiên Colab — không cần tải lại dữ liệu mỗi lần mở notebook.

## 1. Mount Drive + clone repo (chỉ cần làm 1 lần)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/FutureGuidedLearning'
REPO_URL = 'https://github.com/SkyeGunasekaran/FutureGuidedLearning.git'

if not os.path.isdir(PROJECT_DIR):
    !git clone "$REPO_URL" "$PROJECT_DIR"
else:
    print('Repo đã có trong Drive, bỏ qua clone.')

%cd {PROJECT_DIR}/CHBMIT
!pwd

## 2. Cài thêm dependency

Dùng torch đã cài sẵn trên Colab (có CUDA tương ứng runtime) — **không** ép đúng bản
`torch==2.1.1+cu121` của tác giả (`requirements.txt`), vì dễ xung đột với driver CUDA
của máy ảo Colab hiện tại. Chỉ cài các package EEG-xử-lý còn thiếu.

In [ ]:
!pip install -q mne==1.5.1 hickle stft tqdm scikit-learn
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 3. Tải dữ liệu CHB-MIT từ PhysioNet (công khai, không cần đăng nhập)

Chỉ tải các bệnh nhân trong `PATIENTS` dưới đây — file lưu vào
`Dataset/chbXX/chbXX_*.edf`, đúng cấu trúc code đang mong đợi (các file CSV mô tả
`seizure_summary.csv`, `segmentation.csv`... đã có sẵn trong repo, không cần tải thêm).

**Bắt đầu với 1 bệnh nhân (`01`) để kiểm chứng pipeline chạy được** — thêm số vào danh
sách `PATIENTS` khi cần mở rộng (README gợi ý: 1, 3, 5, 10 — khớp với
`patient_list` hardcode trong các script: 1,2,3,5,9,10,13,18,19,20,21,23).

In [ ]:
PATIENTS = ['01']  # vd: ['01', '03', '05', '10'] để tải thêm

for p in PATIENTS:
    dest = f'Dataset/chb{p}'
    if os.path.isdir(dest) and any(f.endswith('.edf') for f in os.listdir(dest)):
        print(f'chb{p} đã có dữ liệu, bỏ qua.')
        continue
    url = f'https://physionet.org/files/chbmit/1.0.0/chb{p}/'
    !wget -r -N -c -np -nH --cut-dirs=3 -A edf -P Dataset "{url}"

!ls Dataset/chb01 | head -5
!du -sh Dataset/chb01

## 4. Stage 1 — Train Teacher (seizure detection)

Lưu model vào `pytorch_models/Patient_1_detection` (theo `teacher_settings.json`,
nằm trong Drive nên bền qua các phiên).

In [ ]:
!python -m exp.seizure_detection --patient 1 --epochs 50 --patience 5

## 5. Stage 2 — Train Baseline (seizure prediction, KHÔNG có FGL)

In [ ]:
!python -m exp.seizure_prediction --patient 1 --model CNN_LSTM --trials 3 --epochs 25

## 6. Stage 3 — Train Student + FGL (seizure prediction, CÓ teacher chỉ dẫn)

Cần Stage 1 đã chạy xong (load lại teacher đã lưu ở `pytorch_models/`).

In [ ]:
!python -m exp.FGL_CHBMIT --patient 1 --epochs 30 --trials 3 --optimizer Adam --alpha 0.5 --temperature 4

## 7. Đối chiếu kết quả

- `Detection_results.txt` — AUC-ROC của teacher (Stage 1)
- `Prediction_results.txt` — AUC-ROC của baseline (Stage 2)
- Kết quả Stage 3 in trực tiếp ra output cell trên, không ghi file riêng — chép lại tay.

Bài báo công bố: **tăng trung bình 44,8% AUC-ROC** trên CHB-MIT (so Student FGL với
Baseline). So sánh bằng: `(AUC_student - AUC_baseline) / AUC_baseline`.

Muốn chạy nhiều bệnh nhân: lặp lại từ Mục 3 (thêm số vào `PATIENTS`), rồi đổi
`--patient 1` thành số bệnh nhân tương ứng ở Mục 4-6 (hoặc `--patient all` để chạy hết
12 bệnh nhân trong danh sách mặc định — sẽ mất nhiều giờ, nên làm từng bệnh nhân một
khi còn đang kiểm chứng).